# European Call Pricing Comparison

Comparison of analytical Black-Scholes, Monte Carlo, and Crank-Nicolson pricing for a vanilla European call.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / "Thesis" / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT / "Thesis"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.black_scholes import black_scholes_call
from src.monte_carlo import monte_carlo_european_call
from src.finite_differences import crank_nicolson_call


In [ ]:
# General option parameters
S0 = 100
K = 110
T = 2
r = 0.035
sigma = 0.35

# Finite-difference grid
S_max = 600
M = 200
N = 200

# Monte Carlo
n_paths = 10000

bs_call = black_scholes_call(S0, K, T, r, sigma)
mc_call = monte_carlo_european_call(S0, K, T, r, sigma, n_paths)
price_cn, S_grid, t_grid_cn, V_cn = crank_nicolson_call(S0, K, T, r, sigma, S_max, M, N)

print("Black-Scholes:", bs_call)
print("Monte Carlo:", mc_call)
print("Crank-Nicolson:", price_cn)


In [ ]:
# Monte Carlo convergence over different path counts
ns = [30, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 50000, 100000, 150000, 500000]
simulated_prices = [
    monte_carlo_european_call(S0, K, T, r, sigma, n)
    for n in ns
]

plt.figure(figsize=(11, 5))
plt.plot([str(val) for val in ns], simulated_prices, marker="o", linestyle="-", label="MC convergence")
plt.axhline(y=bs_call, color="r", linestyle="--", label="Black-Scholes price")
plt.xlabel("Number of simulated paths")
plt.ylabel("Price")
plt.title("Monte Carlo Convergence vs Analytical Black-Scholes Solution")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Pricing across volatility levels
sigmas = [IV / 100 for IV in range(10, 45, 2)]
pricing_across_iv = {
    "analytical_prices": [],
    "MC_prices": [],
    "CN_prices": [],
}

for sigma_i in sigmas:
    pricing_across_iv["analytical_prices"].append(black_scholes_call(S0, K, T, r, sigma_i))
    pricing_across_iv["MC_prices"].append(monte_carlo_european_call(S0, K, T, r, sigma_i, n_paths))
    price_cn_i, _, _, _ = crank_nicolson_call(S0, K, T, r, sigma_i, S_max, M, N)
    pricing_across_iv["CN_prices"].append(price_cn_i)

print(f'The analytical prices are {pricing_across_iv["analytical_prices"]}')
print(f'The MC prices are {pricing_across_iv["MC_prices"]}')
print(f'The Crank-Nicolson prices are {pricing_across_iv["CN_prices"]}')


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(
    sigmas,
    pricing_across_iv["analytical_prices"],
    label="Analytical (Black-Scholes)",
    color="black",
    linestyle="--",
    linewidth=2,
)
plt.scatter(
    sigmas,
    pricing_across_iv["MC_prices"],
    label="Monte Carlo",
    color="blue",
    alpha=0.6,
    s=30,
)
plt.plot(
    sigmas,
    pricing_across_iv["CN_prices"],
    label="Crank-Nicolson",
    color="red",
    alpha=0.8,
)
plt.title("Option Price Comparison: Analytical vs Numerical Methods", fontsize=14)
plt.xlabel(r"Volatility ($\sigma$)", fontsize=12)
plt.ylabel("Call Option Price ($C$)", fontsize=12)
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "pricing_comparison.png")
plt.show()
